# Polymer Markov-Corrected MPC Legacy Prototype (Pre-Unified)

Restored pre-unified polymer Markov prototype from the commit immediately before `ce34e86` so the older implementation can be run side-by-side with the current unified notebook. This notebook delegates to `report/scripts/generate_polymer_markov_correction_assets_legacy.py`, which preserves the older control-loop logic.

## 1. Imports And Configuration

Defaults are loaded from the current polymer baseline configuration. The default run is the existing disturbed profile with `n_tests=200`, `set_points_len=400`, `predict_h=9`, and `cont_h=3`.

In [1]:
from pathlib import Path

from report.scripts.generate_polymer_markov_correction_assets_legacy import (
    build_config,
    run_polymer_markov_correction,
)

CONFIG = build_config()
CONFIG

{'agent_kind': 'td3',
 'run_mode': 'disturb',
 'n_tests': 200,
 'set_points_len': 400,
 'warm_start': 10,
 'test_cycle': [False, False, False, False, False],
 'nominal_qi': 108.0,
 'nominal_qs': 459.0,
 'nominal_ha': 1050000.0,
 'qi_change': 0.85,
 'qs_change': 1.3,
 'ha_change': 0.85,
 'predict_h': 9,
 'cont_h': 3,
 'decision_interval': 1,
 'Q1_penalty': 5.0,
 'Q2_penalty': 1.0,
 'R1_penalty': 1.0,
 'R2_penalty': 1.0,
 'use_shifted_mpc_warm_start': False,
 'basis_family': 'io_pair_gain',
 'z_bound': 0.05,
 'prediction_window': 20,
 'lambda_z': 0.001,
 's_pred_min': 1e-06,
 'gain_drift_max': 0.1,
 'nominal_cost_relative_tol': 0.1,
 'nominal_cost_absolute_tol': 1e-08,
 'run_shadow_only': True,
 'run_adaptive_ls': True,
 'run_live_corrected_mpc': True,
 'run_rl_proposal': True,
 'rl_fallback_to_ls': True,
 'rl_store_executed_action_in_replay': True,
 'rl_save_agent_checkpoint': True,
 'td3_agent': {'seed': 7,
  'actor_hidden': [256, 256],
  'critic_hidden': [256, 256],
  'buffer_size': 1

In [2]:
# Optional smoke-test override. Leave empty for the full current polymer default run.
# Example for a quick development check:
# CONFIG_OVERRIDES = {"max_steps": 40, "make_plots": True, "save_outputs": True}
CONFIG_OVERRIDES = {}

effective_config = build_config(CONFIG_OVERRIDES)
print("run_mode:", effective_config["run_mode"])
print("n_tests:", effective_config["n_tests"])
print("set_points_len:", effective_config["set_points_len"])
print("predict_h:", effective_config["predict_h"])
print("cont_h:", effective_config["cont_h"])
print("basis_family:", effective_config["basis_family"])
print("max_steps:", effective_config["max_steps"])

run_mode: disturb
n_tests: 200
set_points_len: 400
predict_h: 9
cont_h: 3
basis_family: io_pair_gain
max_steps: None


## 2. Load Polymer Model And Baseline Objects

The runner loads `POLYMER_BASELINE_DEFAULTS`, `PolymerCSTR`, scaling artifacts, augmented offset-free matrices, input bounds, setpoint schedules, and the canonical baseline path through the existing polymer helper layer.

## 3. Local Markov Helper Functions

The helper functions are isolated in the legacy report script, not in shared source modules. They implement Markov blocks, lifted Toeplitz maps for absolute input-deviation levels, prediction scoring, gain drift, constrained LS, and lifted MPC cost using the same convention as `MpcSolverGeneral.mpc_opt_fun`.

## 4-8. Run TD3-Assisted Markov MPC

The runner executes the RL-assisted Markov correction loop: lifted-prediction validation, shadow scoring, constrained LS teacher fitting, TD3 Markov action selection, safety-filtered corrected MPC, replay updates, result saving, and standard MPC comparison plotting. TD3 Markov proposals are enabled by default, with constrained LS retained as the warm-start teacher and safety fallback.

In [3]:
RUN_OUTPUT = run_polymer_markov_correction(CONFIG_OVERRIDES)
RUN_OUTPUT["episode_reward_table"]

The system is observable.


C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Simulation\mpc.py:123: UserWarning: Convergence was not reached after maxiter iterations.
You asked for a tolerance of 0.001, we got 0.9999999422182041.
  obs_gain_calc = signal.place_poles(A.T, C.T, desired_poles, method='KNV0')


Sub_Episode: 1 | avg. reward: 231.65391542406152 | accepted: 0.94 | TD3 accepted: 0.0 | LS fallback: 0.0 | nominal fallback: 0.04875 | avg z: [ 0.00105533  0.04129786  0.00398812 -0.01892179]
Sub_Episode: 2 | avg. reward: 229.23816967731665 | accepted: 0.955 | TD3 accepted: 0.0 | LS fallback: 0.0 | nominal fallback: 0.045 | avg z: [ 0.00085481  0.03972835  0.00365434 -0.02087037]
Sub_Episode: 3 | avg. reward: 226.97925193203787 | accepted: 0.96375 | TD3 accepted: 0.0 | LS fallback: 0.0 | nominal fallback: 0.03625 | avg z: [ 0.00094132  0.04015627  0.0037028  -0.02112111]
Sub_Episode: 4 | avg. reward: 225.04526112061657 | accepted: 0.97125 | TD3 accepted: 0.0 | LS fallback: 0.0 | nominal fallback: 0.02875 | avg z: [ 0.00100254  0.040547    0.00360691 -0.02131576]
Sub_Episode: 5 | avg. reward: 222.91046951968855 | accepted: 0.97875 | TD3 accepted: 0.0 | LS fallback: 0.0 | nominal fallback: 0.02125 | avg z: [ 0.00111443  0.04091171  0.00378598 -0.02152405]
Sub_Episode: 6 | avg. reward: 22

,episode,complete_episode,start_step,end_step,nominal_mpc_avg_reward,markov_td3_avg_reward,markov_minus_nominal
0,1,True,0,799,222.885387,231.653915,8.768528
1,2,True,800,1599,221.024187,229.238170,8.213983
2,3,True,1600,2399,219.510011,226.979252,7.469241
3,4,True,2400,3199,217.996873,225.045261,7.048388
4,5,True,3200,3999,216.469357,222.910470,6.441113
...,...,...,...,...,...,...,...
195,196,True,156000,156799,327.654378,315.704083,-11.950295
196,197,True,156800,157599,309.254785,299.249720,-10.005064
197,198,True,157600,158399,291.986405,285.946184,-6.040221
198,199,True,158400,159199,276.542831,273.465980,-3.076851


## 9. Saved Outputs

The run saves its result bundle, Markov diagnostic figures, and CSV summaries under `Polymer/Results/polymer_markov_corrected_mpc/YYYYMMDD_HHMMSS/`. Standard MPC comparison plots are saved under `Polymer/Results/polymer_markov_compare_<run_mode>/YYYYMMDD_HHMMSS/`, and the living report remains at `report/polymer_markov_correction_progress.md`.

In [4]:
print("Report:", RUN_OUTPUT["report_path"])
print("Result folder:", RUN_OUTPUT["result_dir"])
print("Result bundle:", RUN_OUTPUT["result_path"])
print("Comparison folder:", RUN_OUTPUT["comparison_dir"])
print("Figures:")
for path in RUN_OUTPUT["figures"]:
    print(" -", path)

Report: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\report\polymer_markov_correction_progress.md
Result folder: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results\polymer_markov_corrected_mpc\20260510_123506
Result bundle: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results\polymer_markov_corrected_mpc\20260510_123506\input_data.pkl
Comparison folder: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results\polymer_markov_compare_disturb\20260510_144133
Figures:
 - C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results\polymer_markov_corrected_mpc\20260510_123506\phase1_lifted_equivalence.png
 - C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\RL_assisted_MPC\Polymer\Results\polymer_markov_corrected_mpc\20260510_123506\phase2_prediction_score_trace.png
 - C:\Users\HAMEDI\OneDrive -